In [ ]:

# <output_rules>
# Respond with a JSON object.
# Every field is required. Do not skip or add fields.
# Your chosen_position must be a coordinate that appears in can_move_to. No other value is valid.

# <decision>
#   <chosen_position>[x, y] — must appear in can_move_to</chosen_position>
#   <target_agent_group>[citizens / pillars / police / activists]</target_agent_group>
#   <target_pillar_institution>
#   If and only if, the target_agent_group is 'pillars', mention the target institution type, visible in your neighborhood.
#   Write N/A otherwise.
#   </target_pillar_institution>
#   <primary_attribute>[Gandhian attribute]</primary_attribute>
#   <call_to_action_type>[sit_in / march / boycott / strike / civil_disobedience /  petition / hunger_strike]</call_to_action_type>
# </decision>

# </output_rules>



# <decision>
#   <chosen_position>[x, y] — must appear in can_move_to</chosen_position>
#   <target_group>[citizens / pillars / police / activists]</target_group>
#   <target_pillar_institution>
#   If target_group is pillars, name the institution type visible in your neighborhood.
#   Write N/A otherwise.
#   </target_pillar_institution>
#   <primary_attribute>[Gandhian attribute]</primary_attribute>
#   <call_to_action_type>[sit_in / march / boycott / strike / civil_disobedience / petition / hunger_strike]</call_to_action_type>
# </decision>

# </output_rules>



In [2]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from LLM.chat_model import ChatModel

model = ChatModel(
    provider     = "huggingface",
    prompt_dir   = "./Prompts",
    prompt_type  = "ZeroShot",
    use_memory   = False,
    model_name   = "meta-llama/Llama-3.1-8B-Instruct",
    quantization = "4bit",
    device       = "cuda",
)

# Test call with fake context
test_context = {
    "my_position": [10, 10],
    "neighborhood": {
        "north": {
            "can_move_to": [[10, 11], [10, 12]],
            "citizens": {"suffering": 3},
            "pillars": {},
            "police": {},
            "activists": {}
        }
    }
}

import time
t0 = time.time()
result = model.ask_activist_position(test_context)
print(f"Response time: {time.time()-t0:.1f}s")
print(result.decision)



[HuggingFaceProvider] env=local  vram=12.0GB  slurm=False
[HuggingFaceProvider] Loading meta-llama/Llama-3.1-8B-Instruct  quant=4bit  device_map={'': 0}  max_memory={0: '9GB'}


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[HuggingFaceProvider] Model ready.
[HuggingFaceProvider] Constrained decoding unavailable (ImportError) — using unconstrained.
{
  "decision": {
    "chosen_position": [10, 11],
    "target_agent_group": "citizens",
    "target_pillar_institution": "N/A",
    "primary_attribute": "Empathy",
    "call_to_action_type": "civil_disobedience"
  }
}
Response time: 47.5s
chosen_position=[10, 11] target_agent_group='citizens' target_pillar_institution='N/A' primary_attribute='Empathy' call_to_action_type='civil_disobedience'


In [ ]:
from run_simulation import RunOrchestrator, BASE_PARAMS
from Logging.log_config import LogConfig

LOG_CFG = LogConfig(
        provider    = "huggingface",
        prompt_type = "ZeroShot",
        memory      = "stateful",
        vision      = "vision-4",
        base_dir    = "SimulationLogs",
    )

model_params = dict(BASE_PARAMS)


# Run 10 times sequentially
orchestrator = RunOrchestrator(
    base_config  = LOG_CFG,
    params       = model_params,
    max_workers  = 1,       # sequential — safe for API rate limits
    start_run_id = 1,       # change to resume from a specific run
)

summary = orchestrator.run(n_runs=3)   # start with 3 to verify it works

# HF_HOME=C:/Users/fmahnoor1/.cache/huggingface

In [1]:
from run_simulation import RunOrchestrator, BASE_PARAMS
from Logging.log_config import LogConfig

# Define your log config once
LOG_CFG = LogConfig(
    provider    = "gemini_flash",
    prompt_type = "ZeroShot",
    memory      = "stateful",
    vision      = "vision-4",
    base_dir    = "SimulationLogs",
)

# Override any params
model_params = dict(BASE_PARAMS)
# model_params["PillarProxStrat"] = 3

# Run 10 times sequentially
orchestrator = RunOrchestrator(
    base_config  = LOG_CFG,
    params       = model_params,
    max_workers  = 15,       # sequential — safe for API rate limits
    start_run_id = 98,       # change to resume from a specific run
)

summary = orchestrator.run(n_runs=10)



# from 15-64 waste of tokens 




[RunOrchestrator] Starting 10 runs (15 workers)
  Run  102  ✓  resistance    621.3s  [1/10]
  Run  105  ✓  resistance    1178.8s  [2/10]
  Run  101  ✓  resistance    2528.3s  [3/10]
  Run  103  ✓  resistance    2701.4s  [4/10]
  Run  106  ✓  resistance    3284.9s  [5/10]
  Run   98  ✓  resistance    3304.0s  [6/10]
  Run  100  ✓  regime        3598.7s  [7/10]
  Run  104  ✓  regime        5163.4s  [8/10]
  Run   99  ✓  regime        5488.1s  [9/10]
  Run  107  ✓  regime        5974.9s  [10/10]
  Batch summary saved → SimulationLogs\gemini_flash\ZeroShot\stateful\vision-4\batch_summary_20260322_211732.json

  Batch complete: 10 runs in 5980.4s
  Resistance wins :   6 / 10
  Regime wins     :   4 / 10
  Errors          :   0 / 10


In [ ]:
# [RunOrchestrator] Starting 10 runs (10 workers)
#   Run   73  ✓  resistance    540.0s  [1/10]
#   Run   69  ✓  resistance    1427.6s  [2/10]
#   Run   74  ✓  resistance    1933.1s  [3/10]
#   Run   67  ✓  resistance    2707.1s  [4/10]
#   Run   65  ✓  resistance    3305.3s  [5/10]
#   Run   72  ✓  regime        5527.2s  [6/10]
#   Run   66  ✓  regime        5618.8s  [7/10]
#   Run   71  ✓  regime        5794.6s  [8/10]
#   Run   68  ✓  resistance    6020.1s  [9/10]
#   Run   70  ✓  resistance    7762.9s  [10/10]
#   Batch summary saved → SimulationLogs\gemini_flash\ZeroShot\stateful\vision-4\batch_summary_20260317_182813.json

# ============================================================
#   Batch complete: 10 runs in 7781.2s
#   Resistance wins :   7 / 10
#   Regime wins     :   3 / 10
#   Errors          :   0 / 10
# ===============================


# [RunOrchestrator] Starting 10 runs (10 workers)
#   Run   83  ✓  resistance    994.1s  [1/10]
#   Run   84  ✓  resistance    1971.7s  [2/10]
#   Run   77  ✓  resistance    2299.7s  [3/10]
#   Run   75  ✓  resistance    2554.2s  [4/10]
#   Run   80  ✓  resistance    2746.5s  [5/10]
#   Run   81  ✓  regime        4253.0s  [6/10]
#   Run   76  ✓  regime        5352.1s  [7/10]
#   Run   79  ✓  regime        5600.8s  [8/10]
#   Run   78  ✓  resistance    5838.8s  [9/10]
#   Run   82  ✓  resistance    7376.1s  [10/10]
#   Batch summary saved → SimulationLogs\gemini_flash\ZeroShot\stateful\vision-4\batch_summary_20260317_214328.json

# ============================================================
#   Batch complete: 10 runs in 7391.8s
#   Resistance wins :   7 / 10
#   Regime wins     :   3 / 10
#   Errors          :   0 / 10
# ============================================================



# [RunOrchestrator] Starting 3 runs (10 workers)
#   Run   87  ✓  regime        4761.0s  [1/3]
#   Run   85  ✓  resistance    5367.3s  [2/3]
#   Run   86  ✓  regime        8099.1s  [3/3]
#   Batch summary saved → SimulationLogs\gemini_flash\ZeroShot\stateful\vision-4\batch_summary_20260318_001547.json

# ============================================================
#   Batch complete: 3 runs in 8109.8s
#   Resistance wins :   1 / 3
#   Regime wins     :   2 / 3
#   Errors          :   0 / 3
# ============================================================


# [RunOrchestrator] Starting 10 runs (15 workers)
#   Run   18  ✓  resistance    1207.8s  [1/10]
#   Run   24  ✓  resistance    1338.1s  [2/10]
#   Run   22  ✓  resistance    2577.6s  [3/10]
#   Run   15  ✓  resistance    6827.9s  [4/10]
#   Run   23  ✓  regime        6996.1s  [5/10]
#   Run   19  ✓  resistance    7250.9s  [6/10]
#   Run   21  ✓  regime        7683.8s  [7/10]
#   Run   20  ✓  regime        7802.2s  [8/10]
#   Run   17  ✓  regime        8222.5s  [9/10]
#   Run   16  ✓  regime        9922.5s  [10/10]
#   Batch summary saved → SimulationLogs\gemini_flash\ZeroShot\stateful\vision-4\batch_summary_20260318_114046.json

# ============================================================
#   Batch complete: 10 runs in 9963.0s
#   Resistance wins :   5 / 10
#   Regime wins     :   5 / 10
#   Errors          :   0 / 10
# ============================================================

# [RunOrchestrator] Starting 10 runs (15 workers)
#   Run   33  ✓  resistance    1693.1s  [1/10]
#   Run   34  ✓  resistance    1953.3s  [2/10]
#   Run   32  ✓  resistance    1958.2s  [3/10]
#   Run   26  ✓  resistance    2257.7s  [4/10]
#   Run   25  ✓  resistance    2451.9s  [5/10]
#   Run   31  ✓  resistance    2765.9s  [6/10]
#   Run   29  ✓  resistance    3838.4s  [7/10]
#   Run   30  ✓  regime        4146.2s  [8/10]
#   Run   28  ✓  resistance    5127.4s  [9/10]
#   Run   27  ✓  resistance    5205.4s  [10/10]
#   Batch summary saved → SimulationLogs\gemini_flash\ZeroShot\stateful\vision-4\batch_summary_20260318_132339.json

# ============================================================
#   Batch complete: 10 runs in 5219.3s
#   Resistance wins :   9 / 10
#   Regime wins     :   1 / 10
#   Errors          :   0 / 10
# ============================================================


# [RunOrchestrator] Starting 10 runs (15 workers)
#   Run   35  ✓  resistance    2395.0s  [1/10]
#   Run   40  ✓  resistance    2545.7s  [2/10]
#   Run   44  ✓  resistance    2982.9s  [3/10]
#   Run   42  ✓  resistance    3422.5s  [4/10]
#   Run   41  ✓  resistance    4019.5s  [5/10]
#   Run   38  ✓  resistance    4430.4s  [6/10]
#   Run   36  ✓  resistance    4966.0s  [7/10]
#   Run   37  ✓  resistance    5835.3s  [8/10]
#   Run   39  ✓  resistance    6143.9s  [9/10]
#   Run   43  ✓  resistance    9234.3s  [10/10]
#   Batch summary saved → SimulationLogs\gemini_flash\ZeroShot\stateful\vision-4\batch_summary_20260318_181820.json

# ============================================================
#   Batch complete: 10 runs in 9252.5s
#   Resistance wins :  10 / 10
#   Regime wins     :   0 / 10
#   Errors          :   0 / 10
# ============================================================



# [RunOrchestrator] Starting 10 runs (15 workers)
#   Run   58  ✓  resistance    379.1s  [1/10]
#   Run   55  ✓  resistance    442.9s  [2/10]
#   Run   56  ✓  resistance    1731.9s  [3/10]





# [RunOrchestrator] Starting 6 runs (15 workers)
#   Run   60  ✓  resistance    1609.8s  [1/6]
#   Run   62  ✓  regime        3601.8s  [2/6]
#   Run   63  ✓  regime        4013.8s  [3/6]
#   Run   59  ✓  resistance    5592.5s  [4/6]
#   Run   64  ✓  regime        5600.2s  [5/6]
#   Run   61  ✓  regime        5653.9s  [6/6]
#   Batch summary saved → SimulationLogs\gemini_flash\ZeroShot\stateful\vision-4\batch_summary_20260319_142014.json

# ============================================================
#   Batch complete: 6 runs in 5666.4s
#   Resistance wins :   2 / 6
#   Regime wins     :   4 / 6
#   Errors          :   0 / 6
# ============================================================

